In [ ]:
# 1. Libraries
import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

# 2. Load dataset
df = pd.read_csv("Comis-Non-NY-20190207.csv")

# 3. Combine text columns
text_cols = df.select_dtypes(include="object").columns
df["text"] = df[text_cols].fillna("").astype(str).agg(" ".join, axis=1)

# 4. Clean text
df["text"] = df["text"].str.lower()
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", " ", x)
)

# 5. NLP - TF-IDF
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X = vectorizer.fit_transform(df["text"])

# 6. Find best K
scores = []

for k in range(2, 10):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

plt.plot(range(2, 10), scores, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.show()

# 7. K-Means
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 8. Show cluster sizes
print(df["cluster"].value_counts().sort_index())

# 9. Top words in each cluster
words = vectorizer.get_feature_names_out()

for i in range(k):
    top_words = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(words[top_words]))

# 10. Visualise clusters
svd = TruncatedSVD(n_components=2, random_state=42)
X_2D = svd.fit_transform(X)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_2D[:, 0],
    X_2D[:, 1],
    c=df["cluster"],
    alpha=0.6
)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("K-Means NLP Clusters")
plt.show()

# 11. Save results
df.to_csv("Comis-Non-NY-20190207_clustered.csv", index=False)

print("Done!")